In [3]:
!cd /home/songyl/GitHub/phenology-twitter/

In [1]:
# !pip install graphistry # run this line if you have not installed graphistry 
import graphistry

# register for a graphistry account here: https://www.graphistry.com/
graphistry.register(api=3, username='<your username>', password='<your password>')

In [33]:
import pandas as pd

edges = pd.read_csv('../data/processed/network/pollen.csv').head(250)
# edges = edges[edges['count'] >= 50]
print(edges.head())

              to            from  count
0     JoanneFOX5  GoodDayAtlanta      2
1     LiaBurris3         BTS_twt      2
2  OhShit_ImHIGH     yeathatsLEO      2
3        1YungDg          1ankyy      1
4      45Polemic    RadioFreeTom      1


In [34]:
nodes = pd.read_csv('../data/processed/user_type/pollen_labeled.csv').drop('description', axis=1).rename(columns={'user': 'node'})
nodes = nodes.melt(id_vars='node', var_name='type', value_name='value').dropna(subset=['value']).drop('value', axis=1)
nodes['type'] = nodes['type'].replace(['other_organization', 'other_individual'], 'others')
print(nodes.head())

              node   type
0       JoanneFOX5  media
9    AlyssaPejicWx  media
10  AndrewTateKLTV  media
25       BobbyWTSP  media
29            CBS6  media


In [49]:
g = graphistry.edges(edges, 'from', 'to').nodes(nodes, 'node').bind(edge_weight='count')

g = g.encode_point_color('type',
                    categorical_mapping={
        'media': 'orange',
        'expert': 'blue',
        'others': 'silver'
    },
    default_mapping='silver'
                    )\
    .encode_point_icon(
      'type',
      shape="circle", #clip excess
      categorical_mapping={ # https://fontawesome.com/v4/icons/
          'media': 'newspaper-o',
            'expert': 'flask',
            'others': 'user'},
      default_mapping="question")\
    .encode_point_size(
        'type',
        categorical_mapping={
           'media': 5,
            'expert': 5,
            'others': 5
        },
        default_mapping=5
    )

# https://hub.graphistry.com/docs/api/1/rest/url/
URL_PARAMS = {'play': 500, 
              'pointSize': 0.25,
              'pointOpacity': 0.2,
              'edgeCurvature': 0.2,
              'edgeOpacity': 0.2,
              'precisionVsSpeed': -1, 
              'gravity': 2, 
              'scalingRatio': 0.2, 
              'edgeInfluence': 2, 
              'showPointsOfInterest': False,
              'dissuadeHubs': True,
              'linLog': True,
             'pruneOrphans': True,
             'showHistograms':False}
g = g.settings(url_params=URL_PARAMS)

g.plot(render=True)